# Two-model ensemble: and_mask + new_letters on the test surfaces

Same inference path as `test_inference.ipynb` (saved run config replayed, literal surface depth window, surface-anchor
norm, composite cache), run with both checkpoints in cell 2; their maps are combined like `test_inference_ensemble.ipynb`
(mean of z-scores, stretched p1..p99.5). Both models are scaled and stretched with one shared set of statistics pooled
over both maps.

Nothing is displayed; each fragment is saved to `output/test_visualizations/and_letters_ensemble/<name>.jpg`
(extras under `extras/`) as a 2x2 grid at the render's native size:

| and_mask | ensemble |
|---|---|
| new_letters (extra letters) | composite (1 cm scale bar) |

`DEPTH_OFFSET` (cell 2) shifts the literal-surface depth window: -1 reads one slice shallower than the estimate.
Run the setup cells, then any fragment cell. PHerc0009b (a training segment) comes first as a quick check.


In [ ]:
import os

# the two runs; config, architecture and geometry come from each checkpoint's saved config.json
CHECKPOINTS = {
    "and_mask": "/vesuvius/models/archs41/and_mask/final.pth",
    "new_letters": "/vesuvius/models/archs41/new_letters_10/final.pth",
}
# literal-surface depth window shift in slices (0 = surface estimate; -1 = one slice shallower)
DEPTH_OFFSET = 0
OUT_NAME = "and_letters_ensemble"


In [ ]:
# run selection: both checkpoints must share the inference geometry, so one input load serves both
import importlib
import json
import os
import sys
from pathlib import Path

if "/vesuvius" not in sys.path:
    sys.path.insert(0, "/vesuvius")
import utils.config

resolve_run_config = importlib.reload(utils.config).resolve_run_config

# training-only settings that do not change inference
_TRAIN_ONLY = {"extra_inklabels", "extra_label_share", "scrolls", "probe_rois", "vis_scroll_ids",
               "label_free_vis_scroll_ids", "divdis_target_scroll_ids"}
RUNS = {}
for key, path in CHECKPOINTS.items():
    model_path = Path(path).resolve()
    if not model_path.is_file():
        raise FileNotFoundError(f"checkpoint is missing: {model_path}")
    run_config, run_config_path = resolve_run_config(model_path)
    RUNS[key] = {"path": model_path, "config": run_config, "config_path": run_config_path}
    print(f"[run] {key}: {run_config_path} | {model_path}")
MODEL_KEYS = list(RUNS)
_first = RUNS[MODEL_KEYS[0]]["config"]
for key in MODEL_KEYS[1:]:
    other = RUNS[key]["config"]
    data_diff = sorted(k for k in set(_first["data"]) | set(other["data"])
                       if k not in _TRAIN_ONLY and _first["data"].get(k) != other["data"].get(k))
    model_diff = sorted(k for k in set(_first["model"]) | set(other["model"])
                        if _first["model"].get(k) != other["model"].get(k))
    if data_diff or model_diff:
        raise RuntimeError(f"{key} differs from {MODEL_KEYS[0]} at inference: data={data_diff} model={model_diff}")
_RUN_DATA = _first["data"]
_RUN_MODEL = _first["model"]
DEPTH = int(_RUN_DATA["depth"])
MULTITILE_SUBTILE = int(_RUN_MODEL["multitile_subtile"])
MULTITILE_GRID = int(_RUN_MODEL["multitile_grid"])
# inference window step in px (<= MULTITILE_GRID * MULTITILE_SUBTILE); 16 = densest, 32 = ~4x faster
EVAL_STRIDE = 32
_RUN_DATA["eval_stride"] = EVAL_STRIDE
PRELOAD_RAM = True
# test_inference's import cell reports this; this notebook always loads the models
DRY_RUN = False
COMPOSITE_METHOD = "maxproj"
COMPOSITE_D0 = 10
COMPOSITE_D1 = 18
COMPOSITE_DISPLAY = "raw"
COMPOSITE_CLAHE = False
GENERATE_COMPOSITES = False
VOXEL_UM = 9.362
PAD_PX = 24
OUTPUT_DIR = "output"
COMPOSITE_CACHE = "output/composite_cache"
# ensemble stretch percentiles (test_inference_ensemble / ensemble_maps.py)
P_LO, P_HI = 1.0, 99.5
FRAGMENTS = {
    "auto_grown_20260814140748": 20260814140748,
    "auto_grown_20260717193517": 20260717193517,
    "auto_grown_20260720090842": 20260720090842,
    "auto_grown_20250703034159": 20250703034159,
    "auto_grown_20260723112922652_merged": 20260723112922,
    "auto_grown_20260921094413486": 20260921094413,
    "auto_grown_20260918132724424": 20260918132724,
    "auto_grown_20260922073234974": 20260922073234,
    "auto_grown_20260922161631422_merged": 20260922161631,
    "auto_grown_20260925085345806_abf": 20260925085345,
    "PHerc0800_z9264_w040_abf": 20260928000001,
    "PHerc0125_z15344_w040_abf": 20260928000002,
    "PHerc0211_z7312_w080_abf": 20260928000003,
    "PHerc0211_z5520_w040_abf": 20260928000004,
}
# training segments: their curated masks are never regenerated from the mid slice
TRAINING_SEGMENTS = {"pherc0009b": 20250919125754}
EXTRAS_SUBDIR = "extras"
EXTRAS_MESH_ROOT = os.path.join("tifxyz", "atlas")
QUICK_D_MESH_ROOT = os.path.join("tifxyz", "quick_patches_from_D")


def is_extra(scroll_id):
    return not str(scroll_id).isdigit()


def mask_dir_for(scroll_id):
    return os.path.join("masks", EXTRAS_SUBDIR) if is_extra(scroll_id) else "masks"


def mask_path_for(scroll_id):
    return os.path.join(mask_dir_for(scroll_id), f"{scroll_id}.png")


def surface_root_for(config, scroll_id):
    root = config.data.surface_label_dir
    return os.path.join(root, EXTRAS_SUBDIR) if is_extra(scroll_id) else root


if DEPTH != 8 or not _RUN_MODEL.get("surface_teacher_input") or not _RUN_DATA.get("surface_relative_depth_window"):
    raise RuntimeError("selected runs are not literal-surface true slice-8")
if not _RUN_MODEL.get("early_2d_unet") or not _RUN_MODEL.get("gated_stems"):
    raise RuntimeError("selected runs are not early-2D gated models")
if EVAL_STRIDE > MULTITILE_GRID * MULTITILE_SUBTILE:
    raise RuntimeError(f"EVAL_STRIDE={EVAL_STRIDE} exceeds the {MULTITILE_GRID * MULTITILE_SUBTILE}px multitile output")
print(f"models={MODEL_KEYS} depth={DEPTH} offset={DEPTH_OFFSET:+d} stride={EVAL_STRIDE} -> {OUT_NAME}")


In [ ]:
import gc
import importlib
import os
import sys
import types

REPO = os.getenv("VESUVIUS_REPO", "/vesuvius" if os.name == "posix" else r"C:\Users\ChenJeff\Documents\vesuvius")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

from utils.config import Config
from utils.dataloader import _load_unified_cache
from utils.model import create_model
from utils import visualizer as visualizer_module

# reload the bounded-memory row reader in existing kernels
visualizer_module = importlib.reload(visualizer_module)
TBV = visualizer_module.TensorboardVisualizer
group_by_depth = visualizer_module.group_by_depth
predict_tiles = visualizer_module.predict_tiles
load_or_create_midslice_mask = visualizer_module.load_or_create_midslice_mask

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(COMPOSITE_CACHE, exist_ok=True)
print(
    f"repo={REPO}",
    f"| torch={torch.__version__}",
    f"| cuda={torch.cuda.is_available()}",
    f"| dry_run={DRY_RUN}",
    "| bounded-memory visualizer reloaded",
)

In [ ]:
# composite helper and projection settings
def _project_depth(vol, d0, d1, method, bounds=None):
    """project only the requested spatial bounds over slices [d0, d1)"""
    _, height, width = map(int, vol.shape)
    y0, y1, x0, x1 = bounds or (0, height, 0, width)
    d0, d1 = max(0, d0), min(d1, int(vol.shape[0]))
    shape = (y1 - y0, x1 - x0)
    if method == "maxproj":
        acc = np.zeros(shape, np.float32)
        for depth in range(d0, d1):
            acc = np.maximum(acc, np.asarray(vol[depth, y0:y1, x0:x1]).astype(np.float32))
        return acc
    if method == "meanproj":
        acc = np.zeros(shape, np.float64)
        for depth in range(d0, d1):
            acc += np.asarray(vol[depth, y0:y1, x0:x1])
        return (acc / max(d1 - d0, 1)).astype(np.float32)
    if method == "minproj":
        acc = np.full(shape, np.inf, np.float32)
        for depth in range(d0, d1):
            acc = np.minimum(acc, np.asarray(vol[depth, y0:y1, x0:x1]).astype(np.float32))
        return np.where(np.isfinite(acc), acc, 0).astype(np.float32)
    if method == "stdproj":
        total = np.zeros(shape, np.float64)
        total_squared = np.zeros(shape, np.float64)
        for depth in range(d0, d1):
            layer = np.asarray(vol[depth, y0:y1, x0:x1]).astype(np.float64)
            total += layer
            total_squared += layer * layer
        count = max(d1 - d0, 1)
        mean = total / count
        return np.sqrt(np.clip(total_squared / count - mean * mean, 0, None)).astype(np.float32)
    if method == "midslice":
        return np.asarray(vol[(d0 + d1) // 2, y0:y1, x0:x1]).astype(np.float32)
    raise ValueError(f"unknown composite method: {method}")


def _display_map(proj, mask, mode, use_clahe):
    """map a float projection to uint8 for display"""
    if mode == "raw":
        image = np.clip(proj, 0, 255).astype(np.uint8)
    else:
        lo, hi = np.percentile(proj[mask], [1, 99]) if mask.any() else (float(proj.min()), float(proj.max()))
        display = np.clip((proj - lo) / max(hi - lo, 1e-6), 0, 1)
        image = (display * 255).astype(np.uint8)
    if use_clahe:
        image = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image)
    return image * mask.astype(np.uint8)


def make_composite(scroll_id, bounds, method=None, d0=None, d1=None, use_cache=True, display=None, clahe=None):
    """return a uint8 fiber composite restricted to the data bbox"""
    method = method or COMPOSITE_METHOD
    display = display or COMPOSITE_DISPLAY
    clahe = COMPOSITE_CLAHE if clahe is None else clahe
    vol = zarr.open(os.path.join(Config().data.zarr_path, f"{scroll_id}.zarr"), mode="r")
    depth_count = int(vol.shape[0])
    d0 = COMPOSITE_D0 if d0 is None else d0
    d1 = (COMPOSITE_D1 if COMPOSITE_D1 is not None else depth_count) if d1 is None else d1
    d0, d1 = max(0, d0), min(d1, depth_count)
    y0, y1, x0, x1 = bounds
    display_tag = f"_{display}" + ("_clahe" if clahe else "")
    mask_path = mask_path_for(scroll_id)
    if is_extra(scroll_id):
        # extras cache one full-frame composite in composite_cache/extras (R2-synced by assemble_test_segments.py)
        cache_path = os.path.join(COMPOSITE_CACHE, EXTRAS_SUBDIR, f"{scroll_id}_{method}_{d0}-{d1}_full{display_tag}.png")
        if not (use_cache and not GENERATE_COMPOSITES and os.path.exists(cache_path)):
            full_mask = np.array(Image.open(mask_path).convert("L")) > 0
            image = _display_map(_project_depth(vol, d0, d1, method), full_mask, display, clahe)
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            Image.fromarray(image).save(cache_path)
            print(f"[composite] {scroll_id} {method} d{d0}-{d1} full{display_tag} -> {cache_path} {image.shape}")
        return np.array(Image.open(cache_path).convert("L"))[y0:y1, x0:x1]
    bounds_tag = f"y{y0}-{y1}_x{x0}-{x1}"
    cache_path = os.path.join(
        COMPOSITE_CACHE,
        f"{scroll_id}_{method}_{d0}-{d1}_{bounds_tag}{display_tag}.png",
    )
    if use_cache and not GENERATE_COMPOSITES and os.path.exists(cache_path):
        return np.array(Image.open(cache_path).convert("L"))
    full_mask = (np.array(Image.open(mask_path).convert("L")) > 0) if os.path.exists(mask_path) else None
    proj = _project_depth(vol, d0, d1, method, bounds=bounds)
    mask = full_mask[y0:y1, x0:x1] if full_mask is not None else (proj > 0)
    image = _display_map(proj, mask, display, clahe)
    Image.fromarray(image).save(cache_path)
    print(f"[composite] {scroll_id} {method} d{d0}-{d1} {bounds_tag}{display_tag} -> {cache_path} {image.shape}")
    return image


In [ ]:
# inference, shared-statistics ensemble, and the 2x2 figure
from matplotlib.patches import Rectangle
from utils.config import ScrollConfig

from utils.fast_figure import colorize, gray_rgb, save_panel_grid
# masked-out prediction cells (matplotlib PRED_CMAP bad colour)
PRED_BAD_RGB = (0.78, 0.86, 0.95)
_AUXILIARY_PREFIXES = ("supcon_head.", "domain_head.")
# matplotlib/agg and jpeg both cap each image dimension below 2**16 px
MAX_FIGURE_PX = 65000
FIGURE_DPI = 100
# inverted grayscale: 0 (papyrus) = white, 1 (ink) = black; masked-out cells pale blue
PRED_CMAP = plt.get_cmap("gray_r").copy()
PRED_CMAP.set_bad((0.78, 0.86, 0.95))
PANEL_TITLES = {"and_mask": "and_mask", "new_letters": "new_letters (extra letters)"}


def build_config(run_data=None, run_model=None):
    run_data = _RUN_DATA if run_data is None else run_data
    run_model = _RUN_MODEL if run_model is None else run_model
    config = Config()
    # replay the full saved run config so geometry and surface window match training
    for name, value in run_data.items():
        if name == "scrolls":
            config.data.scrolls = [ScrollConfig(**item) for item in value]
        elif name != "probe_rois" and hasattr(config.data, name):
            setattr(config.data, name, value)
    for name, value in run_model.items():
        if hasattr(config.model, name):
            setattr(config.model, name, value)
    config.model.compile_model = False
    config.tra.supcon = False
    config.tra.dann = False
    config.device = "cuda" if torch.cuda.is_available() else "cpu"
    return config


def _checkpoint_state(path):
    state = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    cleaned = {}
    for key, value in state.items():
        key = key.removeprefix("module.").removeprefix("_orig_mod.")
        if not key.startswith(_AUXILIARY_PREFIXES):
            cleaned[key] = value
    return cleaned, len(state) - len(cleaned)


def load_model(key):
    run = RUNS[key]
    config = build_config(run["config"]["data"], run["config"]["model"])
    model, parameter_count = create_model(config)
    state, ignored = _checkpoint_state(run["path"])
    model.load_state_dict(state, strict=True)
    model.eval()
    print(f"loaded {key}: {run['path']} params={parameter_count:,} ignored_training_only={ignored}")
    return model


def load_inputs(config, scroll_id, bounds):
    """load volume, mask, norm, surface maps, and tile coords once for both models"""
    source = zarr.open(os.path.join(config.data.zarr_path, f"{scroll_id}.zarr"), mode="r")
    mask_full = np.array(Image.open(mask_path_for(scroll_id)).convert("L")) > 0
    y0, y1, x0, x1 = bounds
    if PRELOAD_RAM:
        vol = np.asarray(source[:, y0:y1, x0:x1])
        mask = mask_full[y0:y1, x0:x1]
        y_range, x_range = (0, y1 - y0), (0, x1 - x0)
    else:
        vol = source
        mask = mask_full
        y_range, x_range = (y0, y1), (x0, x1)
    norm = get_norm(scroll_id, vol, mask)
    surface_dir = os.path.join(surface_root_for(config, scroll_id), str(scroll_id))
    surface_depth = np.load(os.path.join(surface_dir, "depth.npy"), mmap_mode="r")
    surface_confidence = np.load(os.path.join(surface_dir, "confidence.npy"), mmap_mode="r")
    if PRELOAD_RAM:
        surface_depth = surface_depth[y0:y1, x0:x1]
        surface_confidence = surface_confidence[y0:y1, x0:x1]
    fake = types.SimpleNamespace(c=config)
    coords = TBV._gen_tile_coords(
        fake, (config.data.d_start, config.data.d_end), y_range, x_range, mask, z_step=config.data.depth,
    )
    grouped = group_by_depth(coords)
    if len(grouped) != 1:
        raise RuntimeError(f"surface-relative inference expected one depth pass, got {len(grouped)}")
    depth_offset = next(iter(grouped))
    return {
        "scroll_id": scroll_id, "vol": vol, "mask": mask, "y_range": y_range, "x_range": x_range, "norm": norm,
        "surface_depth": surface_depth, "surface_confidence": surface_confidence,
        "coords": grouped[depth_offset], "depth_start": config.data.d_start + depth_offset,
    }


def predict_map(model, config, inputs, key):
    """literal-surface pass with the window shifted by DEPTH_OFFSET slices"""
    return predict_tiles(
        config, model, inputs["vol"], inputs["mask"], inputs["coords"], inputs["y_range"], inputs["x_range"],
        inputs["depth_start"], f"{key}_{inputs['scroll_id']}_surface{DEPTH_OFFSET:+d}", *inputs["norm"],
        surface_depth_map=inputs["surface_depth"], surface_confidence_map=inputs["surface_confidence"],
        surface_depth_offset=DEPTH_OFFSET,
    )


def ensemble_maps(maps):
    """one shared mean/sd and p1..p99.5 pooled over every model's valid cells; each model is stretched with the
    shared percentiles, and the ensemble is the mean z-score stretched by its own p1..p99.5"""
    valid = np.logical_and.reduce([np.isfinite(pmap) for pmap in maps.values()])
    pooled = np.concatenate([pmap[valid] for pmap in maps.values()]).astype(np.float64)
    if pooled.size == 0:
        blank = np.full(valid.shape, np.nan, np.float32)
        return {key: blank for key in maps}, blank
    mean, sd = float(pooled.mean()), float(pooled.std()) + 1e-6
    lo, hi = (float(v) for v in np.percentile(pooled, [P_LO, P_HI]))
    stretched = {
        key: np.where(valid, np.clip((pmap - lo) / max(hi - lo, 1e-9), 0, 1), np.nan).astype(np.float32)
        for key, pmap in maps.items()
    }
    z = np.mean([np.where(valid, (pmap - mean) / sd, 0.0) for pmap in maps.values()], axis=0)
    ens_lo, ens_hi = (float(v) for v in np.percentile(z[valid], [P_LO, P_HI]))
    ensemble = np.where(valid, np.clip((z - ens_lo) / max(ens_hi - ens_lo, 1e-9), 0, 1), np.nan)
    return stretched, ensemble.astype(np.float32)


def _save_figure(rows, crop_hw, out_path):
    """lay out (title, image, kind) rows on a pixel-exact grid and save it (numpy/OpenCV, one encode)"""
    grid = []
    for row in rows:
        cells = []
        for title, image, kind in row:
            if kind.startswith("composite"):
                cells.append((title, gray_rgb(image), cv2.INTER_AREA, VOXEL_UM if kind == "composite_scale" else None))
            else:
                cells.append((title, colorize(image, "gray_r", PRED_BAD_RGB), cv2.INTER_NEAREST, None))
        grid.append(cells)
    save_panel_grid(grid, crop_hw, out_path, gap_px=PAD_PX)


def render_fragment(name, scroll_id, out_path=None):
    """both models on one input load -> shared-statistics ensemble -> 2x2 grid saved to disk (nothing displayed)"""
    mask_full = np.array(Image.open(mask_path_for(scroll_id)).convert("L"))
    bounds = (0, mask_full.shape[0], 0, mask_full.shape[1])
    crop_height, crop_width = mask_full.shape[:2]
    composite = make_composite(scroll_id, bounds=bounds)
    if composite.shape != (crop_height, crop_width):
        composite = cv2.resize(composite, (crop_width, crop_height), interpolation=cv2.INTER_AREA)
    inputs = load_inputs(C, scroll_id, bounds)
    maps = {key: predict_map(MODELS[key], C, inputs, key) for key in MODEL_KEYS}
    del inputs
    stretched, ensemble = ensemble_maps(maps)
    tag = f"surface {DEPTH_OFFSET:+d}"
    rows = [
        [(f"{PANEL_TITLES['and_mask']}  {tag}", stretched["and_mask"], "pred"),
         (f"ensemble (mean z-score)  {tag}", ensemble, "pred")],
        [(f"{PANEL_TITLES['new_letters']}  {tag}", stretched["new_letters"], "pred"),
         ("composite view", composite, "composite_scale")],
    ]
    out_path = out_path or os.path.join(OUT_DIR, f"{name}.jpg")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    _save_figure(rows, (crop_height, crop_width), out_path)
    print(f"[saved] {out_path} bbox={bounds}")
    del rows, maps, stretched, ensemble
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# build the shared inference config, load both models once
from utils.platform import get_zarr_dir

C = build_config()
C.data.zarr_path = get_zarr_dir()
MODELS = {key: load_model(key) for key in MODEL_KEYS}
OUT_DIR = os.path.join(OUTPUT_DIR, "test_visualizations", OUT_NAME + (f"_d{DEPTH_OFFSET:+d}" if DEPTH_OFFSET else ""))
os.makedirs(OUT_DIR, exist_ok=True)
print("models:", MODEL_KEYS, "| zarr:", C.data.zarr_path, "| norm:", C.data.norm_mode, "| out:", OUT_DIR)


In [ ]:
# fragment readiness: assemble, mask, norm and surface labels on demand (from test_inference)
import subprocess
from utils.norm import compute_norm

MASK_DIR = os.path.join(REPO, "masks")
MASK_DIR = os.path.join(REPO, "masks")
# set by render_extras so helper subprocesses don't print into the notebook
SUBPROCESS_QUIET = False


def _subprocess_output():
    return {"stdout": subprocess.DEVNULL, "stderr": subprocess.DEVNULL} if SUBPROCESS_QUIET else {}


def ensure_assembled(scroll_id):
    """render the test zarr from its tifxyz mesh when it (or its completion mask) is absent"""
    zarr_path = os.path.join(C.data.zarr_path, f"{scroll_id}.zarr")
    # the assembler writes the mask last, so a zarr without a mask is an interrupted render
    if os.path.isdir(zarr_path) and os.path.isfile(os.path.join(REPO, mask_path_for(scroll_id))):
        return
    print(f"[assemble] {scroll_id}: zarr missing or incomplete -> rendering into {C.data.zarr_path}", flush=True)
    subprocess.run(
        [
            sys.executable, os.path.join(REPO, "assemble_test_segments.py"),
            "--only", str(scroll_id),
            "--out-dir", C.data.zarr_path,
        ] + (["--include-extras"] if is_extra(scroll_id) else []),
        cwd=REPO,
        check=True,
        **_subprocess_output(),
    )
    if not os.path.isdir(zarr_path):
        raise FileNotFoundError(f"assembly did not produce {zarr_path}")


def ensure_normalized(scroll_id):
    """populate norm_cache.json for this fragment when its stats are absent"""
    stats = _load_unified_cache().get(str(scroll_id))
    if stats and all(key in stats for key in ("mean", "std", "min", "max")):
        return
    print(f"[norm] {scroll_id}: stats missing -> computing", flush=True)
    compute_norm(scroll_id, C.data.zarr_path, mask_dir=os.path.join(REPO, mask_dir_for(scroll_id)))


def ensure_surface_supervision(scroll_id):
    """generate literal surface labels when depth/confidence maps are absent"""
    surface_root = surface_root_for(C, scroll_id)
    surface_dir = os.path.join(surface_root, str(scroll_id))
    if all(os.path.isfile(os.path.join(surface_dir, name)) for name in ("depth.npy", "confidence.npy")):
        return
    print(f"[surface] {scroll_id}: labels missing -> generating in {surface_dir}", flush=True)
    review_dir = os.path.join("output", "surface_review") + (f"/{EXTRAS_SUBDIR}" if is_extra(scroll_id) else "")
    subprocess.run(
        [
            sys.executable, os.path.join(REPO, "generate_surface_supervision.py"),
            "--scroll-id", str(scroll_id),
            "--z-start", "4", "--z-end", "28",
            "--zarr-dir", C.data.zarr_path,
            "--mask-dir", os.path.join(REPO, mask_dir_for(scroll_id)),
            "--output-dir", surface_root,
            "--review-dir", review_dir,
        ],
        cwd=REPO,
        check=True,
        **_subprocess_output(),
    )


def ensure_fragment_ready(scroll_id):
    """assemble, refresh the mask, normalize, and build surface labels as needed"""
    ensure_assembled(scroll_id)
    volume = zarr.open(os.path.join(C.data.zarr_path, f"{scroll_id}.zarr"), mode="r")
    mask = load_or_create_midslice_mask(
        volume, os.path.join(REPO, mask_path_for(scroll_id)), refresh=True
    )
    ensure_normalized(scroll_id)
    ensure_surface_supervision(scroll_id)
    return mask



In [ ]:
# surface-anchor norm readiness override for future models
import json
from utils.norm import SURFACE_ANCHOR_CACHE_PATH, UNIFIED_CACHE_PATH, compute_norm, compute_surface_anchors, load_cached_norm

def ensure_surface_norm_ready(scroll_id):
    mode = str(getattr(C.data, "norm_mode", "global"))
    if mode != "surface_anchor":
        return
    sid = str(scroll_id)
    cache = {}
    if os.path.exists(SURFACE_ANCHOR_CACHE_PATH):
        try:
            with open(SURFACE_ANCHOR_CACHE_PATH, "r", encoding="utf-8") as handle:
                loaded = json.load(handle)
            if isinstance(loaded, dict):
                cache = loaded
        except Exception:
            cache = {}
    if sid in cache:
        return
    cache[sid] = compute_surface_anchors(
        sid,
        C.data.zarr_path,
        surface_dir=os.path.join(REPO, surface_root_for(C, scroll_id)),
        mask_dir=os.path.join(REPO, mask_dir_for(scroll_id)),
    )
    temporary = f"{SURFACE_ANCHOR_CACHE_PATH}.tmp"
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(cache, handle, indent=2)
    os.replace(temporary, SURFACE_ANCHOR_CACHE_PATH)
    print(f"[anchor] {sid} {cache[sid]}", flush=True)

def get_norm(scroll_id, vol, mask):
    del vol, mask
    mode = str(getattr(C.data, "norm_mode", "global"))
    if mode == "surface_anchor":
        ensure_surface_norm_ready(scroll_id)
    stats = load_cached_norm(scroll_id, UNIFIED_CACHE_PATH, mode=mode)
    if stats is None:
        compute_norm(
            scroll_id,
            C.data.zarr_path,
            UNIFIED_CACHE_PATH,
            mask_dir=os.path.join(REPO, mask_dir_for(scroll_id)),
        )
        if mode == "surface_anchor":
            ensure_surface_norm_ready(scroll_id)
        stats = load_cached_norm(scroll_id, UNIFIED_CACHE_PATH, mode=mode)
    if stats is None:
        raise RuntimeError(f"normalization cache missing for {scroll_id} (mode={mode})")
    return stats

def ensure_fragment_ready(scroll_id):
    """assemble, refresh the mask, normalize, and build surface labels as needed"""
    ensure_assembled(scroll_id)
    volume = zarr.open(os.path.join(C.data.zarr_path, f"{scroll_id}.zarr"), mode="r")
    mask = load_or_create_midslice_mask(
        volume, os.path.join(REPO, mask_path_for(scroll_id)),
        refresh=int(scroll_id) not in TRAINING_SEGMENTS.values() if str(scroll_id).isdigit() else True,
    )
    ensure_normalized(scroll_id)
    ensure_surface_supervision(scroll_id)
    ensure_surface_norm_ready(scroll_id)
    return mask

In [ ]:
# --- PHerc0009b (training segment, quick check) ---
NAME = "pherc0009b"
ensure_fragment_ready(TRAINING_SEGMENTS[NAME])
render_fragment(NAME, TRAINING_SEGMENTS[NAME])


In [ ]:
# --- test fragment 1: PHerc0813 (updated patch) ---
NAME = "auto_grown_20260814140748"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- test fragment 2: PHerc0211 merged ---
NAME = "auto_grown_20260717193517"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- test fragment 3: PHerc1203 ---
NAME = "auto_grown_20260720090842"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- test fragment 4: PHerc1447 ---
NAME = "auto_grown_20250703034159"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- test fragment 5: PHerc0826 merged ---
NAME = "auto_grown_20260723112922652_merged"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- test fragment 6: PHerc0846A ---
NAME = "auto_grown_20260921094413486"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- test fragment 7: PHerc0175A ---
NAME = "auto_grown_20260918132724424"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- test fragment 8: PHerc0306B ---
NAME = "auto_grown_20260922073234974"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- test fragment 9: PHerc0800 merged ---
NAME = "auto_grown_20260922161631422_merged"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- test fragment 10: PHerc1447 second surface ---
NAME = "auto_grown_20260925085345806_abf"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- test fragment 11: PHerc0800 expanded auto-winder segment ---
NAME = "PHerc0800_z9264_w040_abf"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- test fragment 12: PHerc0125 expanded auto-winder segment ---
NAME = "PHerc0125_z15344_w040_abf"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- PHerc0211 auto-winder segment z7312 ---
NAME = "PHerc0211_z7312_w080_abf"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- PHerc0211 auto-winder segment z5520 ---
NAME = "PHerc0211_z5520_w040_abf"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME])


In [ ]:
# --- extras setup: atlas + quick_patches_from_D -> <out>/extras/<name>.jpg (no display) ---
import contextlib
import io

# plain-text bar: tqdm.auto's ipywidget bar fails in the VS Code notebook renderer
from tqdm import tqdm

EXTRAS_OUT_DIR = os.path.join(OUT_DIR, EXTRAS_SUBDIR)
EXTRA_IDS = sorted(
    name for name in (os.listdir(EXTRAS_MESH_ROOT) if os.path.isdir(EXTRAS_MESH_ROOT) else [])
    if os.path.isfile(os.path.join(EXTRAS_MESH_ROOT, name, "x.tif"))
)
# quick_patches_from_D/<scroll>/<segment>
QUICK_D_IDS = {
    scroll: sorted(
        name for name in os.listdir(os.path.join(QUICK_D_MESH_ROOT, scroll))
        if os.path.isfile(os.path.join(QUICK_D_MESH_ROOT, scroll, name, "x.tif"))
    )
    for scroll in sorted(os.listdir(QUICK_D_MESH_ROOT))
}


def render_extras(scroll, extra_ids=None, assemble=False):
    """render every extra segment of one scroll (atlas by name prefix unless extra_ids is given);
    assemble=True renders missing zarrs from S3 instead of skipping them"""
    global SUBPROCESS_QUIET
    if extra_ids is None:
        extra_ids = [extra_id for extra_id in EXTRA_IDS if extra_id.split("_")[0] == scroll]
    done, skipped, failed = 0, [], []
    SUBPROCESS_QUIET = True
    try:
        for extra_id in tqdm(extra_ids, desc=scroll, unit="patch"):
            assembled = (os.path.isdir(os.path.join(C.data.zarr_path, f"{extra_id}.zarr"))
                         and os.path.isfile(mask_path_for(extra_id)))
            if not (assembled or assemble):
                skipped.append(extra_id)
                continue
            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    ensure_fragment_ready(extra_id)
                    render_fragment(extra_id, extra_id, out_path=os.path.join(EXTRAS_OUT_DIR, f"{extra_id}.jpg"))
                done += 1
            except Exception as exc:
                failed.append((extra_id, str(exc)))
    finally:
        SUBPROCESS_QUIET = False
    print(f"[extras] {scroll}: saved {done}/{len(extra_ids)} -> {EXTRAS_OUT_DIR}; "
          f"not assembled: {len(skipped)}; failed: {len(failed)}")
    for extra_id, message in failed:
        print(f"  FAILED {extra_id}: {message}")


print(f"{len(EXTRA_IDS)} atlas segments:", {s: sum(i.startswith(s + "_") for i in EXTRA_IDS)
                                            for s in sorted({i.split('_')[0] for i in EXTRA_IDS})})
print("quick_patches_from_D segments:", {scroll: len(ids) for scroll, ids in QUICK_D_IDS.items()})


In [ ]:
# --- quick_patches_from_D: PHerc0846A -> <out>/extras/<name>.jpg ---
render_extras("PHerc0846A", QUICK_D_IDS["PHerc0846A"], assemble=True)


In [ ]:
# --- quick_patches_from_D: PHerc0175A -> <out>/extras/<name>.jpg ---
render_extras("PHerc0175A", QUICK_D_IDS["PHerc0175A"], assemble=True)


In [ ]:
# --- extras: PHerc0211 (51 segments) ---
render_extras("PHerc0211")


In [ ]:
# --- extras: PHerc0257 (1 segment) ---
render_extras("PHerc0257")


In [ ]:
# --- extras: PHerc0358 (2 segments) ---
render_extras("PHerc0358")


In [ ]:
# --- extras: PHerc0800 (44 segments) ---
render_extras("PHerc0800")


In [ ]:
# --- extras: PHerc0813 (46 segments) ---
render_extras("PHerc0813")


In [ ]:
# --- extras: PHerc0826 (2 segments) ---
render_extras("PHerc0826")
